# 17 — Interpretabilidade visual e detecção de objetos

**EP · Data Science Knowledge Base | Deep Learning | Notebook 17/20**


**Pré-requisitos:** Python, NumPy, fundamentos de aprendizado supervisionado; JupyterLab; TensorFlow/Keras apenas nos laboratórios indicados.

**Escopo de validação:** os exemplos básicos podem ser reproduzidos sem TensorFlow; laboratórios opcionais precisam de validação local independente.

---

**Deep Learning · Capítulo de estudo independente**

> Material didático autoral · estudo independente · exemplos locais e laboratórios opcionais

**Referência curricular:** `material histórico de referência`. O material original foi preservado; este capítulo é uma reconstrução didática independente.

### Materiais complementares

- [Guia Técnico](./reference_materials/17_interpretabilidade_visao.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_interpretabilidade_visao.pdf)


## Como estudar este capítulo

1. Leia a teoria e deduza as expressões apresentadas.
2. Execute as células do **núcleo obrigatório** em um kernel limpo.
3. Confira a interpretação e responda ao exercício antes de abrir a solução.
4. Se tiver TensorFlow e Keras compatíveis, execute o **laboratório opcional**.

**Pré-requisitos transversais:** Python (01), Estatística (03), preparação/validação de ML (05); aplicações textuais remetem à PLN (06). Não há necessidade de repetir integralmente esses cursos.

## Objetivos de aprendizagem

- Entender saliência, mapas de ativação e distinguir classificação de detecção.
- Relacionar a técnica à sua função de perda, representação e protocolo de avaliação.
- Explicar limitações e formular um teste reprodutível.

## 1. Fundamentação conceitual

Mapas de saliência mostram sensibilidade local e não causalidade. Detecção de objetos combina classe e localização, frequentemente por caixas delimitadoras; segmentação estima regiões por pixel. Avaliação exige IoU, precisão e recall por limiar.

**Fonte histórica de apoio:** notas do curso original, consultadas apenas como referência de contexto; as explicações e exemplos desta edição são independentes.

**Questões críticas:** quais pressupostos são necessários? O que a função aprende? Qual métrica indica utilidade real? Que tipo de vazamento ou má interpretação invalidaria a conclusão?

## 2. Matemática essencial

**Expressão esquemática:** `IoU=área(A∩B)/área(A∪B)`

A expressão acima resume a relação central. Para a derivação, identifique dimensões, hipóteses e operações; não interprete notação compacta como uma implementação completa.

## 3. Experimento obrigatório — NumPy e/ou scikit-learn

Este experimento é propositalmente pequeno: **executa sem GPU, internet ou serviço externo**. Quando há treino de baseline, não interpretá-lo como evidência da qualidade de uma rede neural.

In [1]:
def iou(a,b):
 x1=max(a[0],b[0]);y1=max(a[1],b[1]);x2=min(a[2],b[2]);y2=min(a[3],b[3])
 inter=max(0,x2-x1)*max(0,y2-y1)
 area=lambda q:(q[2]-q[0])*(q[3]-q[1])
 return inter/(area(a)+area(b)-inter)
print('IoU:',round(iou((0,0,2,2),(1,1,3,3)),3))
assert abs(iou((0,0,2,2),(1,1,3,3))-1/7)<1e-9

IoU: 0.143


### Interpretação e limitações

Compare o resultado obtido com a formulação matemática. Confirme shapes, escala e hipótese de modelagem. **Um experimento sintético valida uma operação, não substitui avaliação em dados reais.** Ao treinar modelos, separe dados e ajuste transformações apenas no treino; preserve um conjunto final de teste.

## 4. Laboratório neural opcional — TensorFlow / Keras 3

Esta seção demonstra **APIs reais**, mas depende de versões compatíveis e pode exigir mais recursos que o núcleo. Não é necessária para executar o notebook básico. Não se deve confundir um teste de chamada de API com avaliação experimental de desempenho. Se o pacote TensorFlow não estiver disponível, será exibido um aviso de não execução.

In [2]:
# Laboratório opcional: execute após instalar e configurar TensorFlow/Keras compatíveis.
# Se TensorFlow estiver ausente, o exemplo é ignorado com aviso explícito.
import importlib.util
if importlib.util.find_spec('tensorflow') is None:
    print('Laboratório TensorFlow/Keras não executado: TensorFlow ausente neste kernel.')
else:
    import keras
    from keras import layers
    entrada=keras.Input(shape=(32,32,3));features=layers.Conv2D(8,3,activation='relu')(entrada)
    model=keras.Model(entrada,features);print('Mapa intermediário:',model.output_shape)

I0000 00:00:1794602174.262598   31277 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Mapa intermediário: (None, 30, 30, 8)


## 5. Exercício de fixação

**Questão:** Uma heatmap de saliência prova causalidade?

<details>
<summary>Confira a solução comentada</summary>

**Solução:** Não; destaca regiões sensíveis ao modelo, não necessariamente evidência causal.

**Verificação:** explique o resultado usando notação, shapes e consequências para generalização.
</details>

## 6. Erros comuns e boas práticas

- Confundir ajuste no treino com generalização para dados não vistos.
- Reportar somente uma métrica ou uma execução aleatória.
- Ignorar shape, tipo de variável e convenção do backend.
- Executar exemplos com pesos baixados sem registrar versão, licença e procedência.
- Confundir demonstração matemática com validação de um modelo de produção.

## 7. Resumo de consulta rápida

**Ideia central:** Mapas de saliência mostram sensibilidade local e não causalidade.

**Expressão:** `IoU=área(A∩B)/área(A∪B)`.

**Lembrete:** começar por baseline; respeitar separação de dados; documentar decisões; avaliar limitações.

**Ponte com outros cursos:** ML I (05) para métricas e validação; PLN (06) para modelos de linguagem; projetos multidisciplinares ficam reservados à fase posterior.

**Fontes de atualização (consultar quando necessário):** [Keras 3](https://keras.io/guides/), [TensorFlow](https://www.tensorflow.org/guide), [scikit-learn](https://scikit-learn.org/stable/user_guide.html).

## Laboratório aplicado revisado — Mapa de saliência numérica com gradientes de entrada

Mapas de saliência localizam sensibilidades de predição, mas não estabelecem relação causal; separar interpretação de detecção.

**Procedimento:** execute a próxima célula com o núcleo declarado em `requirements.txt`, confira os `assert` e interprete os resultados. O experimento é independente dos exemplos anteriores.

In [3]:
import numpy as np
img=np.array([[.1,.2,.3],[.4,.5,.6],[.7,.8,.9]],dtype=float)
weights=np.array([[0.,1.,0.],[0.,2.,0.],[0.,1.,0.]])
def score(x):return float(np.sum(x*weights))
h=1e-5
saliency=np.zeros_like(img)
for i in range(img.shape[0]):
 for j in range(img.shape[1]):
    p=img.copy();q=img.copy();p[i,j]+=h;q[i,j]-=h
    saliency[i,j]=(score(p)-score(q))/(2*h)
assert np.allclose(saliency,weights,atol=1e-8)
print('Saliência numérica:\n',saliency)
print('Saliência é sensibilidade local, NÃO uma explicação causal.')

Saliência numérica:
 [[0. 1. 0.]
 [0. 2. 0.]
 [0. 1. 0.]]
Saliência é sensibilidade local, NÃO uma explicação causal.


### Exportação de resultados (local)

Execute esta célula após o experimento acima. Os arquivos são salvos em `outputs/` e não entram automaticamente no Git.

In [4]:
# Exportação de um gráfico didático: saliência numérica, não explicação causal.
from pathlib import Path
import matplotlib.pyplot as plt

out=Path('outputs'); out.mkdir(exist_ok=True)
fig,ax=plt.subplots(figsize=(5,4))
im=ax.imshow(saliency,cmap='viridis')
ax.set(title='Saliência numérica (exemplo didático)',xlabel='Coluna',ylabel='Linha')
fig.colorbar(im,ax=ax,label='Sensibilidade local')
fig.tight_layout()
caminho=out/'17_mapa_saliencia_numerica.png'
fig.savefig(caminho,dpi=160,bbox_inches='tight')
plt.close(fig)
print('Exportado:',caminho)


Exportado: outputs/17_mapa_saliencia_numerica.png


### Interpretação, limites e verificação

Para um modelo real use gradientes da pontuação da classe em relação à imagem, comparando com abordagens e perturbações alternativas.

**Checklist:** conferir entradas e saídas, sementes, inexistência de uso dos dados de teste durante ajuste e distinção entre uma demonstração sintética e desempenho em aplicações reais.